### Allscripts Sunrise (SCM) - Observation Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur (observations)
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence (links to patients)

**Strategy:**
- Link observations to patients via OrderTaskOccuranceGUID → ClientGUID
- Map ObsItemGUID to OMOP observation concepts
- Use ValueText for text observations
- Filter to Active observations only

**Note:**
- Some observations may not link to order tasks (direct entry)
- These will be skipped in this initial implementation

In [ ]:
source = 'allscripts_scm'

# Transformation

In [ ]:
silver_observation_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(obs_concept.omop_concept_id, 0) AS observation_concept_id,
  CAST(obs.ETL_LOAD_TS AS DATE) AS observation_date,
  obs.ETL_LOAD_TS AS observation_datetime,
  44818701 AS observation_type_concept_id,  -- EHR
  NULL AS value_as_number,
  obs.ValueText AS value_as_string,
  NULL AS value_as_concept_id,
  NULL AS qualifier_concept_id,
  0 AS unit_concept_id,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT('{source}', ' | ', obs.GUID) AS observation_source_value,
  0 AS observation_source_concept_id,
  obs.UnitOfMeasure AS unit_source_value,
  NULL AS qualifier_source_value,
  obs.ValueText AS value_source_value,
  NULL AS observation_event_id,
  NULL AS obs_event_field_concept_id,
  '{source}' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
  ON obs.OrderTaskOccuranceGUID = oto.GUID
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', CHAR(31), 'dbo_cv3client', CHAR(31), 'guid', CHAR(31), CAST(oto.ClientGUID AS BIGINT)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept obs_concept
  ON obs_concept.source_id = CAST(obs.ObsItemGUID AS STRING)
  AND obs_concept.domain_id = 'Observation'
  AND obs_concept.source_system = '{source}'
WHERE obs.GUID IS NOT NULL
  AND oto.ClientGUID IS NOT NULL
  AND obs.StatusType = 1  -- Performed
LIMIT 10000
''')

display(silver_observation_df)
silver_observation_df.createOrReplaceTempView("silver_observation")

# Merge to Silver

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.observation AS t
USING (
  SELECT * FROM silver_observation 
  WHERE observation_date IS NOT NULL
) AS s
ON t.observation_source_value = s.observation_source_value

WHEN MATCHED THEN UPDATE SET
  t.person_id = s.person_id,
  t.observation_concept_id = s.observation_concept_id,
  t.observation_date = s.observation_date,
  t.observation_datetime = s.observation_datetime,
  t.observation_type_concept_id = s.observation_type_concept_id,
  t.value_as_number = s.value_as_number,
  t.value_as_string = s.value_as_string,
  t.value_as_concept_id = s.value_as_concept_id,
  t.qualifier_concept_id = s.qualifier_concept_id,
  t.unit_concept_id = s.unit_concept_id,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.observation_source_concept_id = s.observation_source_concept_id,
  t.unit_source_value = s.unit_source_value,
  t.qualifier_source_value = s.qualifier_source_value,
  t.value_source_value = s.value_source_value

WHEN NOT MATCHED THEN INSERT (
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id,
  source_system
)
VALUES (
  s.person_id,
  s.observation_concept_id,
  s.observation_date,
  s.observation_datetime,
  s.observation_type_concept_id,
  s.value_as_number,
  s.value_as_string,
  s.value_as_concept_id,
  s.qualifier_concept_id,
  s.unit_concept_id,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.observation_source_value,
  s.observation_source_concept_id,
  s.unit_source_value,
  s.qualifier_source_value,
  s.value_source_value,
  s.observation_event_id,
  s.obs_event_field_concept_id,
  s.source_system
);

# Populate Mapping Table

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
  observation_source_value,
  active_flag,
  created_at,
  updated_at
)
SELECT 
  observation_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP()
FROM _exponent.omop_silver.observation
WHERE observation_source_value NOT IN (
  SELECT observation_source_value 
  FROM _exponent.omop_mapping.source_to_observation
  WHERE active_flag = TRUE
);

# Merge to Gold

In [ ]:
%sql
MERGE INTO _exponent.omop.observation AS gold
USING (
  SELECT 
    source_to_observation.observation_id,
    s.person_id,
    s.observation_concept_id,
    s.observation_date,
    s.observation_datetime,
    s.observation_type_concept_id,
    s.value_as_number,
    s.value_as_string,
    s.value_as_concept_id,
    s.qualifier_concept_id,
    s.unit_concept_id,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.observation_source_value,
    s.observation_source_concept_id,
    s.unit_source_value,
    s.qualifier_source_value,
    s.value_source_value
  FROM _exponent.omop_silver.observation s
  JOIN _exponent.omop_mapping.source_to_observation
    ON source_to_observation.observation_source_value = s.observation_source_value
    AND source_to_observation.active_flag = TRUE
) AS src
ON gold.observation_id = src.observation_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.observation_concept_id = src.observation_concept_id,
  gold.observation_date = src.observation_date,
  gold.observation_datetime = src.observation_datetime,
  gold.observation_type_concept_id = src.observation_type_concept_id,
  gold.value_as_number = src.value_as_number,
  gold.value_as_string = src.value_as_string,
  gold.value_as_concept_id = src.value_as_concept_id,
  gold.qualifier_concept_id = src.qualifier_concept_id,
  gold.unit_concept_id = src.unit_concept_id,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.observation_source_value = src.observation_source_value,
  gold.observation_source_concept_id = src.observation_source_concept_id,
  gold.unit_source_value = src.unit_source_value,
  gold.qualifier_source_value = src.qualifier_source_value,
  gold.value_source_value = src.value_source_value

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value
);